In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import  seaborn as sn
import numpy as np

# Partie A : Audit et nettoyage

In [8]:
df = pd.read_csv("../data/credits_microfinance.csv")
print(f"\n shape : {df.shape} ")
print(f"\n columns : {df.columns.to_list() } ")
print(f"Manquants : {df.isnull().sum().sum()} valeurs / {df.isnull().mean().mean()*100:.1f}% du total")
print(f"Types     : \n{df.dtypes.value_counts()}")
print(f"info : {df.info()}")
print("\n description :", df.describe())


 shape : (9255, 29) 

 columns : ['id_client', 'date_octroi', 'age', 'sexe', 'situation_matrimoniale', 'nb_personnes_charge', 'niveau_education', 'zone_habitation', 'secteur_activite', 'anciennete_activite_mois', 'revenu_mensuel_fcfa', 'charges_mensuelles_fcfa', 'possede_compte_epargne', 'montant_epargne_fcfa', 'montant_credit_fcfa', 'duree_credit_mois', 'taux_interet_annuel', 'objet_credit', 'type_garantie', 'membre_groupe_solidaire', 'nb_credits_anterieurs', 'nb_retards_anterieurs', 'score_mobile_money', 'nb_transactions_mm_mois', 'distance_agence_km', 'agent_credit', 'nb_relances_recouvrement', 'statut_dossier', 'defaut_paiement'] 
Manquants : 3307 valeurs / 1.2% du total
Types     : 
object     11
int64       9
float64     9
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9255 entries, 0 to 9254
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id_client   

## A1 Audit des valeurs manquantes

**Question :** les manquants sont-ils aléatoires (MCAR) ou liés à un profil de client
(MAR / MNAR) ? La réponse conditionne la stratégie d'imputation de la partie D.

Méthode : pour chaque colonne trouée, on construit un drapeau `manquant` et on teste
l'indépendance entre ce drapeau et toutes les autres variables  χ² pour les
catégorielles, Mann-Whitney pour les numériques  puis contre la cible `defaut_paiement`.


In [13]:
tableau = pd.DataFrame({
    "n_manquants": df.isna().sum(),
    "taux_%": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str),
})
tableau = tableau[tableau["n_manquants"] > 0].sort_values("taux_%", ascending=False)

print(f"Cellules vides   : {df.isna().sum().sum() / df.size * 100:.2f} % du tableau")
print(f"Lignes complètes : {df.notna().all(axis=1).mean()*100:.1f} %  "
      f"(supprimer les lignes trouées coûterait {(1-df.notna().all(axis=1).mean())*100:.0f} % du jeu)")
display(tableau)

Cellules vides   : 1.23 % du tableau
Lignes complètes : 68.6 %  (supprimer les lignes trouées coûterait 31 % du jeu)


,n_manquants,taux_%,dtype
score_mobile_money,1183,12.78,float64
montant_epargne_fcfa,874,9.44,float64
revenu_mensuel_fcfa,563,6.08,float64
anciennete_activite_mois,343,3.71,float64
niveau_education,204,2.20,object
nb_personnes_charge,140,1.51,float64


fonction de diagnostic

In [14]:
from scipy.stats import chi2_contingency, mannwhitneyu

COLS_CAT = ["zone_habitation", "niveau_education", "secteur_activite", "sexe",
            "situation_matrimoniale", "type_garantie", "objet_credit"]
COLS_NUM = ["age", "revenu_mensuel_fcfa", "charges_mensuelles_fcfa", "montant_credit_fcfa",
            "duree_credit_mois", "distance_agence_km", "nb_transactions_mm_mois",
            "nb_credits_anterieurs", "anciennete_activite_mois"]


def diagnostic_manquants(df, col, cols_cat=COLS_CAT, cols_num=COLS_NUM, alpha=0.05):
    """Teste si l'absence de `col` est liée au profil du client.

    Catégorielles -> chi2 d'indépendance.
    Numériques    -> Mann-Whitney (test de rang : robuste aux sentinelles -999 / 9999
                     et aux erreurs d'unité encore présentes à ce stade).
    Retourne le tableau des p-values, avec correction de Bonferroni.
    """
    mask = df[col].isna()
    print("=" * 78)
    print(f" {col} — {mask.sum()} manquants ({mask.mean()*100:.1f} %)")
    print("=" * 78)

    lignes = []
    for c in cols_cat:
        if c == col:
            continue
        tab = pd.crosstab(df[c], mask)
        if tab.shape[0] < 2 or tab.shape[1] < 2:
            continue
        lignes.append({"variable": c, "type": "cat", "p_value": chi2_contingency(tab)[1]})

    for c in cols_num:
        if c == col:
            continue
        a = df.loc[~mask, c].dropna()
        b = df.loc[mask, c].dropna()
        if len(b) < 10:
            continue
        lignes.append({"variable": c, "type": "num", "p_value": mannwhitneyu(a, b).pvalue})

    res = pd.DataFrame(lignes).sort_values("p_value").reset_index(drop=True)
    n_tests = len(res)
    seuil_bonf = alpha / n_tests                      # correction pour tests multiples
    res["p_value"] = res["p_value"].round(4)
    res["brut"] = np.where(res["p_value"] < alpha, "LIE", "indep.")
    res["corrige"] = np.where(res["p_value"] < seuil_bonf, "LIE", "indep.")
    print(f"{n_tests} tests  seuil brut {alpha}, seuil Bonferroni {seuil_bonf:.4f}")
    print(res.to_string(index=False))

    med = df.groupby(mask)[[c for c in cols_num if c != col]].median().T
    med.columns = ["complet", "manquant"]
    med["ecart_%"] = ((med["manquant"] / med["complet"] - 1) * 100).round(1)
    print("\nMédianes comparées :")
    print(med.round(1).to_string())

    cible = df.groupby(mask)["defaut_paiement"].agg(["count", "mean"])
    cible["mean"] = (cible["mean"] * 100).round(1)
    cible.index = ["complet", "manquant"]
    p_cible = chi2_contingency(pd.crosstab(df["defaut_paiement"], mask))[1]
    print(f"\nTaux de défaut (%) :\n{cible.to_string()}")
    print(f"p = {p_cible:.4f} -> l'absence est "
          f"{'INFORMATIVE' if p_cible < alpha else 'non informative'} sur la cible\n")
    return res


In [15]:
resultats = {}
for col in tableau.index:
    resultats[col] = diagnostic_manquants(df, col)

 score_mobile_money — 1183 manquants (12.8 %)
16 tests  seuil brut 0.05, seuil Bonferroni 0.0031
                variable type  p_value   brut corrige
           type_garantie  cat   0.0580 indep.  indep.
 charges_mensuelles_fcfa  num   0.0945 indep.  indep.
        niveau_education  cat   0.0949 indep.  indep.
  situation_matrimoniale  cat   0.1283 indep.  indep.
anciennete_activite_mois  num   0.1767 indep.  indep.
     montant_credit_fcfa  num   0.2751 indep.  indep.
       duree_credit_mois  num   0.2831 indep.  indep.
      distance_agence_km  num   0.3394 indep.  indep.
     revenu_mensuel_fcfa  num   0.4538 indep.  indep.
                    sexe  cat   0.4858 indep.  indep.
   nb_credits_anterieurs  num   0.6502 indep.  indep.
                     age  num   0.7150 indep.  indep.
            objet_credit  cat   0.8412 indep.  indep.
         zone_habitation  cat   0.8444 indep.  indep.
 nb_transactions_mm_mois  num   0.8972 indep.  indep.
        secteur_activite  cat   0.9440 

In [17]:
print("Le manque est-il lié à la détention d'un compte ?")
display(pd.crosstab(df["possede_compte_epargne"], df["montant_epargne_fcfa"].isna(),
                    normalize="index").round(3))

print("\nQuelle valeur prend l'épargne selon la détention d'un compte ?")
display(df.groupby("possede_compte_epargne")["montant_epargne_fcfa"]
          .agg(["count", "min", "median", "max"]))

co = df[tableau.index].isna().astype(int).corr().round(3)
display(co)

Le manque est-il lié à la détention d'un compte ?


montant_epargne_fcfa,False,True
possede_compte_epargne,,
0,0.902,0.098
1,0.912,0.088



Quelle valeur prend l'épargne selon la détention d'un compte ?


,count,min,median,max
possede_compte_epargne,,,,
0,5286,0.0,0.0,0.0
1,3095,1000.0,23700.0,445300.0


,score_mobile_money,montant_epargne_fcfa,revenu_mensuel_fcfa,anciennete_activite_mois,niveau_education,nb_personnes_charge
score_mobile_money,1.000,0.006,-0.008,-0.012,0.002,0.008
montant_epargne_fcfa,0.006,1.000,-0.000,-0.011,0.004,-0.001
revenu_mensuel_fcfa,-0.008,-0.000,1.000,-0.002,-0.014,0.002
anciennete_activite_mois,-0.012,-0.011,-0.002,1.000,0.013,-0.001
niveau_education,0.002,0.004,-0.014,0.013,1.000,-0.013
nb_personnes_charge,0.008,-0.001,0.002,-0.001,-0.013,1.000


### A1 — Conclusion

68,6 % des lignes seulement sont complètes : supprimer les lignes trouées coûterait
31 % du jeu. L'imputation est donc obligatoire  reste à savoir laquelle.

**~96 tests d'indépendance ont été menés.** Cinq ressortent significatifs au seuil brut
de 5 % (`sexe` p=0,040 ; `anciennete_activite_mois` p=0,031 ; `distance_agence_km`
p=0,042 ; `nb_transactions_mm_mois` p=0,044 ; `situation_matrimoniale` p=0,033), soit
exactement le nombre de faux positifs attendus par le hasard. **Aucun ne survit à la
correction de Bonferroni** (seuil 0,003). Il n'existe donc aucun lien démontrable entre
l'absence d'une valeur et le profil du client.

| Colonne | Taux | Nature | Preuve mesurée | Décision d'imputation |
|---|---|---|---|---|
| `score_mobile_money` | 12,8 % | MCAR | 16 tests, aucun significatif après correction ; médianes identiques (revenu 79 100 vs 77 600, âge 38 vs 38) ; défaut 15,3 % vs 17,2 %, p=0,113 | médiane, **+ indicateur binaire à tester en C** |
| `montant_epargne_fcfa` | 9,4 % | MCAR pour le manque, **valeur déductible** | trous uniformes (9,8 % / 8,8 %) mais épargne = 0 sur 5 286/5 286 lignes sans compte | **0 si pas de compte, médiane du groupe (23 700) sinon** |
| `revenu_mensuel_fcfa` | 6,1 % | MCAR | défaut 15,6 % vs 14,6 %, p=0,541 — écart en sens *inverse* de l'hypothèse MNAR | médiane, apprise dans le pipeline |
| `anciennete_activite_mois` | 3,7 % | MCAR | défaut 15,6 % vs 15,2 %, p=0,895 | médiane |
| `niveau_education` | 2,2 % | MCAR | défaut 15,6 % vs 15,7 %, p=1,000 | modalité la plus fréquente |
| `nb_personnes_charge` | 1,5 % | MCAR | défaut 15,5 % vs 20,7 %, p=0,115 (n=140, non significatif) | médiane |

Les indicateurs d'absence sont par ailleurs **non corrélés entre eux** (|r| < 0,02) :
aucune cause commune de type formulaire tronqué ou agent défaillant.

**Conclusion.** Contrairement à ce que l'intitulé du TP suggérait, les clients sans score
mobile money ne forment aucun profil identifiable : ils ne sont ni plus ruraux
(34,0 % vs 32,7 %), ni moins éduqués, ni plus pauvres. Le mécanisme est MCAR, ce qui
autorise une imputation simple **sans risque de biais**   à condition qu'elle soit
apprise dans le pipeline (partie D2) et non sur le jeu complet.

Deux réserves consignées :
1. Ce diagnostic utilise les catégories non harmonisées (`Rurale` / `RURALE` comptent
   pour deux modalités), ce qui gonfle les degrés de liberté du χ². **À relancer après A4.**
2. Les deux seuls écarts non négligeables sur la cible   `score_mobile_money`
   (17,2 % vs 15,3 %) et `nb_personnes_charge` (20,7 % vs 15,5 %)   sont statistiquement
   non significatifs mais suffisamment marqués pour justifier de tester des indicateurs
   binaires d'absence en partie C, avec/sans, à modèle égal.
